# Reproducing Johnson et al. (2017) on CRCNS pfc-5

**Paper.** Johnson, Dewar, Solbakk, Endestad, Meling & Knight (2017), "Bidirectional
Frontoparietal Oscillatory Systems Support Working Memory", Current Biology 27, 1829-1835,
doi:10.1016/j.cub.2017.05.046. The claims file (`claims/johnson-2017.yaml`) was written from the
version of record with its supplemental information; it decides what is tested.

**Dataset.** CRCNS pfc-5 (doi:10.6080/K0ZC811B), read anonymously from the public bucket
`crcnsarchive` under `pfc-5/live/` (`crcns:pfc-5/unversioned`). CRCNS data sets have no versions;
the files under `live/` are the only published state of the data (the descriptor is at its
version 0.8, April 2021, and `analysis_scripts.tar.gz` holds the 0.61 update of the PSI scripts),
so they are what is pinned. The first code cell prints the MD5 sums from the data set's
`checksums.md5` for every file read, so that a later change is detectable. Each subject's
`.tar.gz` is streamed once; `data_derived.mat` (preprocessed, surface Laplacian, lesions swapped
to the left) and the trial tables are read from it in memory, and the raw `.bdf` is skipped.

**Method.** Measures follow the claims file; where it leaves a detail open, the paper's code
deposited in the data set settles it (the claims file allows this), and where neither does the
readings are all run and the choice adopted is stated (helper docstring, `frontoparietal.py`).

| claim | tested |
|---|---|
| cohort-and-trials | tested |
| demographics-matched | tested |
| patient-factors-no-effect | tested |
| table-s1-patient-accuracy | tested |
| accuracy-patients-impaired | tested |
| patients-above-chance | tested |
| accuracy-no-hemifield-condition-effect | tested |
| erp-lesioned-maintenance | tested |
| erp-processing-interaction | tested |
| erp-intact-none | tested |
| baseline-spectral-interaction | tested |
| baseline-no-group-effect | tested |
| power-anterior-theta-controls | tested |
| power-posterior-alphabeta | tested |
| power-lowtheta-interaction | tested |
| power-no-group-main-effect | tested |
| psi-deltatheta-bottomup-encoding | tested |
| psi-deltatheta-pfc-lead-controls | tested |
| psi-deltatheta-group-clusters | tested |
| psi-network-degree | tested |
| psi-alphabeta-bottomup | tested |
| psi-alphabeta-no-group-effect | tested |

No claim is blocked.


In [ ]:
import time, resource, io, tarfile, re, warnings
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import frontoparietal as fp

T0 = time.time()
print("dataset", fp.DATASET, "doi", fp.DATASET_DOI)
md5 = {}
for line in fp.fetch("checksums.md5").decode().splitlines():
    if line.strip():
        h, name = line.split(None, 1)
        md5[name.strip()] = h
files_read = (["checksums.md5 (itself not listed)", "docs/summary_file_behavior_pfc.txt",
               "docs/summary_file_behavior_ctrl.txt", "analysis_scripts.tar.gz"]
              + [f"data/{s}.tar.gz" for s in fp.SUBJECTS])
for f in files_read:
    print(md5.get(f, "-" * 32), f)

The paper's code, deposited in the data set (`analysis_scripts.tar.gz`). The lines below are
the ones this notebook follows where the claims file leaves a detail open, and the two places
where the code disagrees with the Methods or with the data.

In [ ]:
scripts = {}
with tarfile.open(fileobj=io.BytesIO(fp.fetch("analysis_scripts.tar.gz")), mode="r:gz") as tf:
    for m in tf.getmembers():
        if m.isfile() and m.name.endswith(".m") and "/._" not in m.name:
            scripts[m.name.split("analysis_scripts/")[-1]] = tf.extractfile(m).read().decode("latin-1")
print(sorted(scripts))
def show(name, pattern):
    for i, line in enumerate(scripts[name].splitlines(), 1):
        if re.search(pattern, line):
            print(f"{name}:{i}: {line.strip()}")
show("tfr_stat_group.m", r"neighbours|minnbchan|numrandomization|clusteralpha|tail|layout|method =")
show("tfr_stat_group_hem.m", r"cfgl.channel|cfgr.channel|individual-|powspctrm-")
show("tfr_compute.m", r"cfgc.trials|cfgv.trials|ft_selectdata\(cfgv, proc\)|foi|t_ftimwin|pad|toi_res =")
show("subfunctions/zbaseline.m", r"randsample|ntrials =")
show("psi_compute.m", r"subtracterp|toi_res =|threshold|cfggt.method|fmin = |psianalysis\(")
show("subfunctions/psianalysis.m", r"csd\(|ppsub\(|psisub\(|nulldist\(:|psimtx\(")
show("psi_stat_group.m", r"for e = 1:nchan|subspctrm =")
show("fft_compute.m", r"method|foi|pad")
show("erp_compute.m", r"lpfreq|padding|cfgc.trials|ft_selectdata\(cfgv, proc\)")

In [ ]:
t_load = time.time()
R = fp.process_all(fp.SUBJECTS, workers=4)
failed = {s: r["error"] for s, r in R.items() if "error" in r}
print("subjects loaded", len(R) - len(failed), "failed", failed)
print(f"loading and per-subject computation took {(time.time() - t_load) / 60:.1f} min; "
      f"peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
CTRL = [s for s in fp.CONTROLS if s not in failed]
PFC = [s for s in fp.PATIENTS if s not in failed]
labels = R[CTRL[0]]["label"]
L = {c: i for i, c in enumerate(labels)}
ADJ = fp.neighbours(labels)
ADJ_HEM = ADJ[np.ix_([L[c] for c in fp.LEFT], [L[c] for c in fp.LEFT])]
pfc_doc = pd.read_csv(io.BytesIO(fp.fetch("docs/summary_file_behavior_pfc.txt")), sep="\t")
ctrl_doc = pd.read_csv(io.BytesIO(fp.fetch("docs/summary_file_behavior_ctrl.txt")), sep="\t")

In [ ]:
# What was loaded, and the description checked against the data
rows = []
for s in CTRL + PFC:
    r = R[s]
    csv = r["csv"]
    ti = r["encmain_trialinfo"]
    names = [m[0].split("/")[-1] for m in r["members"] if m[1] > 0]
    # lesioned visual hemifield in the trial table's coding (1 left, 2 right)
    link = fp.link_trials(csv, ti)
    m = csv.iloc[link[link >= 0]]
    ti_l = ti[link >= 0]
    les_vhf = sorted(set(m["vhf"].values[ti_l[:, 2] == 2]))
    same_order = bool(np.array_equal(r["proc_trialinfo"][:, 0], ti[:, 0]))
    same_set = set(r["proc_trialinfo"][:, 0]) == set(ti[:, 0])
    rows.append({
        "subject": s, "csv_trials": len(csv), "derived_trials": len(ti),
        "csv_blocks": csv["block"].nunique(), "linked": int((link >= 0).sum()),
        "info_agrees": bool((m["correct"].values == ti_l[:, 1]).all() and (m["condition"].values == ti_l[:, 4]).all()
                            and (m["jitter"].values == ti_l[:, 5]).all() and (m["rt"].values == ti_l[:, 6]).all()),
        "lesioned_vhf_csv": les_vhf, "pre_eq_enc_info": bool(np.array_equal(r["pretrial_trialinfo"], ti)),
        "proc_same_set": same_set, "proc_same_order": same_order,
        "len_pre_enc_proc": (r["pretrial_lengths"], r["encmain_lengths"], r["proc_lengths"]),
        "fsample_pre_enc_proc": (round(r["pretrial_fsample"], 2), r["encmain_fsample"], r["proc_fsample"]),
        "two_blocks": any("_2" in n for n in names), "channel_replace": any("channel_replace" in n for n in names),
        "neighbour_corr": round(r["neighbour_corr"], 2),
    })
san = pd.DataFrame(rows).set_index("subject")
with pd.option_context("display.width", 250, "display.max_columns", 30, "display.max_rows", 50):
    print(san)
print("\nsubjects: controls", len(CTRL), "patients", len(PFC))
print("channel order of the files differs from ctrl01's in", sum(R[s]["label_file"] != fp.CANON for s in CTRL + PFC),
      "subjects (an interpolated channel moved to the end); every array is put in one order:", all(R[s]["label"] == fp.CANON for s in CTRL + PFC))
print("segment start times (s):", sorted({x for s in CTRL + PFC for k in ("pretrial", "encmain", "proc") for x in R[s][f"{k}_time_first"]}))
print("segment samples pretrial/encmain/proc:", sorted({str(v) for v in san["len_pre_enc_proc"]}),
      "-> at 256 Hz: 0 to 0.5 s, 0 to 1.5 s, 0 to 0.898 s; no buffers")
print("fsample fields:", sorted({str(v) for v in san["fsample_pre_enc_proc"]}))
print("derived trials linked to one trial-table row (trial number and response time):",
      int(san["linked"].sum()), "of", int(san["derived_trials"].sum()), "| trial info agrees where linked:", san["info_agrees"].all())
print("subjects with two trial tables (two recording blocks):", list(san.index[san["csv_blocks"] > 1]))
print("pretrial and encmain trialinfo identical:", san["pre_eq_enc_info"].all())
print("proc holds the same trials as encmain:", san["proc_same_set"].all(),
      "| in the same order:", int(san["proc_same_order"].sum()), "of", len(san))
fing = pd.Series({s: R[s]["fingerprint"] for s in CTRL + PFC})
md5d = pd.Series({s: R[s]["derived_md5"] for s in CTRL + PFC})
print("duplicate recordings (identical first-trial sums or identical data_derived.mat):",
      int(fing.duplicated().sum()), int(md5d.duplicated().sum()))
# hemifield coding and the swap
print("trial-table visual hemifield (1 left, 2 right) of the trials data_derived codes 2 ('lesioned'):",
      sorted({str(v) for v in san["lesioned_vhf_csv"]}), "in every subject")
pat_hem = pfc_doc.set_index("subid")["lesion_hem"]
print("so derived column 3 equals the trial table's hemifield in every trial, for left- and right-lesioned patients alike")
# early visual lateralisation (N1, 150-220 ms): left minus right posterior, column 3 = 2 minus = 1
t_enc = np.arange(385) / fp.FS
w_n1 = (t_enc >= 0.15) & (t_enc < 0.22)
pl4 = [L[c] for c in ("P7", "PO7", "P9", "O1")]
pr4 = [L[c] for c in ("P8", "PO8", "P10", "O2")]
def n1_lat(s):
    d = lambda e: e[pl4][:, w_n1].mean() - e[pr4][:, w_n1].mean()
    return d(R[s]["erp_enc_2"]) - d(R[s]["erp_enc_1"])
n1 = pd.Series({s: n1_lat(s) for s in CTRL + PFC})
for nm, grp in (("controls", CTRL), ("left-lesioned patients", [s for s in PFC if pat_hem[s] == "L"]),
                ("right-lesioned patients", [s for s in PFC if pat_hem[s] == "R"])):
    print(f"N1 lateralisation positive in {int((n1[grp] > 0).sum())} of {len(grp)} {nm}")
print("neighbouring-channel correlation, median (range):", san["neighbour_corr"].median(),
      san["neighbour_corr"].min(), san["neighbour_corr"].max())
# accuracy in the docs summary against the trial tables, under both codings of the hemifield
def acc_vhf(s, v):
    csv = R[s]["csv"]
    return csv.loc[csv.vhf == v, "correct"].mean()
doc = pd.concat([pfc_doc.set_index("subid"), ctrl_doc.set_index("subid")])
cmp = pd.DataFrame({s: {"acc_vhf1": acc_vhf(s, 1), "acc_vhf2": acc_vhf(s, 2), "doc_lesion": doc.loc[s, "vhf_lesion"],
                        "doc_intact": doc.loc[s, "vhf_intact"]} for s in CTRL + PFC}).T
cmp["lesion_is_vhf1"] = (cmp.acc_vhf1 - cmp.doc_lesion).abs().le(0.006) & (cmp.acc_vhf2 - cmp.doc_intact).abs().le(0.006)
cmp["lesion_is_vhf2"] = (cmp.acc_vhf2 - cmp.doc_lesion).abs().le(0.006) & (cmp.acc_vhf1 - cmp.doc_intact).abs().le(0.006)
print(cmp.round(3).to_string())
for nm, grp in (("controls", CTRL), ("patients", PFC)):
    print(f"{nm}: docs 'vhf_lesion' equals the trial-table accuracy of hemifield 1 in {int(cmp.loc[grp, 'lesion_is_vhf1'].sum())}, "
          f"of hemifield 2 in {int(cmp.loc[grp, 'lesion_is_vhf2'].sum())} of {len(grp)} (within 0.006)")
print("docs summary: patients", len(pfc_doc), "controls", len(ctrl_doc),
      "| patient males", int((pfc_doc.gender == "M").sum()), "| control males", int((ctrl_doc.gender == "M").sum()))
# units and ranges
erp_all = np.stack([R[s]["erp_enc_2"] for s in CTRL + PFC])
fft_all = np.stack([R[s]["fft"] for s in CTRL + PFC])
print(f"ERP (data units, no unit stored; microvolts per area under the Laplacian): {erp_all.min():.2f} to {erp_all.max():.2f}")
print(f"baseline power: {fft_all.min():.3g} to {fft_all.max():.3g}")
zz = np.stack([fp.power_z(R[s], "proc", 2) for s in CTRL + PFC])
print(f"power Z (proc, lesioned hemifield, reading pad): {np.nanmin(zz):.1f} to {np.nanmax(zz):.1f}")
m_fit = R[CTRL[0]]["tfr_pre_mask"]
print("frequencies with any baseline value under FieldTrip's window rule ('fit'):",
      fp.FREQS[m_fit.any(1)].min(), "to 40 Hz; proc frequencies with any value:",
      fp.FREQS[R[CTRL[0]]["tfr_proc_mask"].any(1)].min(), "to 40 Hz")
for band in ("dt", "ab"):
    print(f"PSI {band}: time points whose windows all fit, pretrial/encmain/proc:",
          int(R[CTRL[0]][f"psi_{band}_pre_mask"].all(0).sum()), int(R[CTRL[0]][f"psi_{band}_enc_mask"].all(0).sum()),
          int(R[CTRL[0]][f"psi_{band}_proc_mask"].all(0).sum()))

**Do the segments share the clock of the trial table?** The first shape appears at time 0 of
the encoding-maintenance segment and the second at 200 ms, so a visual response should
arise in posterior channels after about 80 ms and again after about 280 ms, and not before.
Below, the posterior ERP (P7, P8, P9, P10, PO7, PO8, O1, O2), both hemifields, tested
against zero across the 34 subjects at each time point; the threshold is the 95th
percentile of the largest |t| when each subject's sign is flipped at random (1000 times).
Each subject's ERP is taken relative to its mean over the first 50 ms, as the pretrial
baseline lies 1.5 s earlier and the start screen leaves an offset.
The trial tables give no event inside the processing segment other than its start.

In [ ]:
post = [L[c] for c in ("P7", "P8", "P9", "P10", "PO7", "PO8", "O1", "O2")]
def first_response(key, n_t):
    x = np.stack([(R[s][f"{key}_1"] + R[s][f"{key}_2"])[post].mean(0) / 2 for s in CTRL + PFC])[:, :n_t]
    x = x - x[:, :13].mean(1, keepdims=True)          # change from the first 50 ms
    tv = stats.ttest_1samp(x, 0).statistic
    rng_clock = np.random.default_rng(11)
    nul = np.array([np.abs(stats.ttest_1samp(x * rng_clock.choice([-1, 1], len(x))[:, None], 0).statistic).max()
                    for _ in range(1000)])
    return tv, np.quantile(nul, 0.95)
tv, thr = first_response("erp_enc", 385)
tt = np.arange(len(tv)) / fp.FS
sig = np.flatnonzero(np.abs(tv) > thr)
print(f"encoding-maintenance: threshold |t| > {thr:.2f}; first time above it {tt[sig[0]] * 1000:.0f} ms; "
      f"peak |t| {np.abs(tv).max():.1f} at {tt[np.abs(tv).argmax()] * 1000:.0f} ms")
for a, b in ((0, 0.06), (0.06, 0.12), (0.12, 0.2), (0.2, 0.28), (0.28, 0.4), (0.4, 0.8), (0.8, 1.5)):
    w = (tt >= a) & (tt < b)
    print(f"  {a * 1000:4.0f}-{b * 1000:4.0f} ms: max |t| = {np.abs(tv[w]).max():.1f}")
tv, thr = first_response("erp_proc", 231)
tt = np.arange(len(tv)) / fp.FS
sig = np.flatnonzero(np.abs(tv) > thr)
print(f"processing: threshold {thr:.2f}; first time above it "
      f"{(tt[sig[0]] * 1000 if len(sig) else float('nan')):.0f} ms; peak |t| {np.abs(tv).max():.1f} at {tt[np.abs(tv).argmax()] * 1000:.0f} ms")

## Behaviour

The hemifield codes disagree (data check above): `data_derived.mat` codes visual hemifield 2
as "lesioned" (descriptor), and that code equals the trial table's "right" in every trial and
every subject; the docs summary file, whose patient values are the paper's Table S1, gives
the patients' lesioned-hemifield accuracy as that of trial-table hemifield 1 in 9 of 14
patients (all those with one trial table). Two codings are therefore run where the lesioned
hemifield matters:

- **A (descriptor)**: hemifield 2 of `data_derived.mat` (trial table "right") is the lesioned
  one in every subject.
- **B (left/right)**: the code is left/right; the lesioned hemifield is the one contralateral
  to the lesion before the swap, right for left-lesioned patients and left for
  right-lesioned patients; controls as in A (which 10 controls were swapped is not in the
  data, so their lesioned side cannot be known).

For behaviour, the docs reading (**S1**: trial-table hemifield 1 is the lesioned one, for
every patient) is also shown, as it is the one that reproduces Table S1.

In [ ]:
pinfo = pfc_doc.set_index("subid")
def les_code(s, coding):
    """Trial-table hemifield (1 left, 2 right) taken as lesioned."""
    if coding == "A":
        return 2
    if coding == "B":
        return 1 if (s in PFC and pinfo.loc[s, "lesion_hem"] == "R") else 2
    if coding == "S1":
        return 1 if s in PFC else 2
def cells(s, coding, source="table"):
    """Accuracy and trial counts per (lesion/intact, condition); source 'table' = every trial of
    the trial tables (after the eye-gaze exclusion), 'derived' = the trials kept in
    data_derived.mat."""
    if source == "table":
        d = R[s]["csv"][["correct", "vhf", "condition"]]
    else:
        ti = R[s]["encmain_trialinfo"]
        d = pd.DataFrame({"correct": ti[:, 1], "vhf": ti[:, 2], "condition": ti[:, 4]})
    lv = les_code(s, coding)
    out = []
    for v, vn in ((lv, 1), (3 - lv, 0)):
        for c in (1, 2, 3):
            x = d.loc[(d.vhf == v) & (d.condition == c), "correct"]
            out.append({"subject": s, "group": int(s in PFC), "lesion_vhf": vn, "condition": c,
                        "k": int(x.sum()), "n": len(x), "acc": x.mean()})
    return out
beh = {cd: pd.DataFrame([row for s in CTRL + PFC for row in cells(s, cd)]) for cd in ("A", "B", "S1")}
print("cells per coding:", {k: len(v) for k, v in beh.items()}, "| empty cells:", int((beh["A"].n == 0).sum()))
print("trials per cell, min to max:", beh["A"].n.min(), "to", beh["A"].n.max())

### cohort-and-trials
> The study has 14 unilateral PFC patients and 20 age- and education-matched controls; after
> cleaning, subjects kept on average 187 trials, 94 +- 21 (55 to 119) in the lesioned and
> 93 +- 22 (51 to 120) in the intact visual hemifield.

In [ ]:
print("patients", len(PFC), "controls", len(CTRL))
rows = []
for s in CTRL + PFC:
    ti = R[s]["encmain_trialinfo"]
    for cd in ("A", "B"):
        lv = les_code(s, cd)
        for rd, sel in (("all kept", np.ones(len(ti), bool)), ("correct", ti[:, 1] == 1)):
            rows.append({"subject": s, "coding": cd, "reading": rd, "total": int(sel.sum()),
                         "lesioned": int((sel & (ti[:, 2] == lv)).sum()), "intact": int((sel & (ti[:, 2] == 3 - lv)).sum())})
tc = pd.DataFrame(rows)
summ = tc.groupby(["reading", "coding"]).agg(
    total_mean=("total", "mean"), les_mean=("lesioned", "mean"), les_sd=("lesioned", lambda x: x.std(ddof=1)),
    les_min=("lesioned", "min"), les_max=("lesioned", "max"), int_mean=("intact", "mean"),
    int_sd=("intact", lambda x: x.std(ddof=1)), int_min=("intact", "min"), int_max=("intact", "max"))
print(summ.round(1).to_string())
print("ranges widened by 10% of each bound: lesioned 49.5 to 130.9, intact 45.9 to 132")
print("\nby group, all kept trials (coding A):")
print(tc[(tc.reading == "all kept") & (tc.coding == "A")].assign(group=lambda d: np.where(d.subject.str.startswith("pfc"), "patients", "controls"))
      .groupby("group")[["total", "lesioned", "intact"]].agg(["mean", "min", "max"]).round(1).to_string())

**Result: reproduced.** 14 patients and 20 controls. Over all trials kept in `data_derived.mat` (reading a) subjects average 187.3 trials, 94.4 +- 21.1 (55 to 119) in the lesioned and 93.0 +- 22.1 (51 to 120) in the intact hemifield under coding A, the paper's numbers to the trial; coding B gives 93.7 +- 21.5 and 93.6 +- 21.8 with the same ranges. Correct trials only (reading b) give 172.4 per subject, outside 187 +- 18.7.

### demographics-matched
> Controls do not differ from patients in age or education, nor in variance.

In [ ]:
age_p, age_c = pfc_doc.test_age.astype(float), ctrl_doc.test_age.astype(float)
edu_p, edu_c = pfc_doc.edu_yrs.astype(float), ctrl_doc.edu_yrs.astype(float)
print(f"age: patients {age_p.mean():.1f} +- {age_p.std():.1f} [{age_p.min():.0f}-{age_p.max():.0f}], "
      f"controls {age_c.mean():.1f} +- {age_c.std():.1f} [{age_c.min():.0f}-{age_c.max():.0f}]")
print(f"education: patients {edu_p.mean():.1f} +- {edu_p.std():.1f}, controls {edu_c.mean():.1f} +- {edu_c.std():.1f}")
print(f"males: patients {(pfc_doc.gender == 'M').sum()}, controls {(ctrl_doc.gender == 'M').sum()}")
for nm, a, b in (("age", age_p, age_c), ("education", edu_p, edu_c)):
    t = stats.ttest_ind(a, b, equal_var=True)
    F = a.var(ddof=1) / b.var(ddof=1)
    print(f"{nm}: t(32) = {t.statistic:.2f}, p = {t.pvalue:.3f}; variance F(13,19) = {F:.2f}, one-tailed p = {stats.f.sf(F, 13, 19):.3f}")

**Result: reproduced.** No test gives p < 0.05: age t(32) = 0.35, p = 0.725 (paper t < 0.36, p > 0.72); education t(32) = -1.38, p = 0.178; variance F(13,19) = 0.76 (age, p = 0.685) and 1.42 (education, p = 0.237). The education test differs from the paper's bound (t < 1.22, p > 0.23), and the education variance test from F < 1.31, p > 0.31; the ages (patients 46.4 +- 16.4, controls 44.1 +- 18.8) match. The demographics come from the docs summary files.

### patient-factors-no-effect
> In patients, no lesion or demographic factor (lesion hemisphere, etiology, size, years since
> lesion, age, gender, education, IQ) nor visual hemifield or condition has a significant
> effect on accuracy, which permits pooling the patients into one group.

Between-subject terms as Table S2 lists them; factors coded +-1 (Type III), covariates as
recorded in `docs/summary_file_behavior_pfc.txt`; etiology is stroke against tumour
resection (Results: "a single stroke or surgical resection of a low-grade tumor").

In [ ]:
pi_ = pinfo.loc[PFC]
hem = np.where(pi_.lesion_hem == "L", 1.0, -1.0)
eti = np.where(pi_.lesion_etiology == "Stroke", 1.0, -1.0)
gen = np.where(pi_.gender == "M", 1.0, -1.0)
B = np.column_stack([hem, eti, pi_.lesion_size, pi_.elapsed_yrs, pi_.test_age, gen, pi_.edu_yrs, pi_.iq,
                     hem * eti, hem * gen, eti * gen]).astype(float)
bnames = ["Hemisphere", "Etiology", "Size", "Elapsed", "Age", "Gender", "Education", "IQ",
          "Hemisphere x Etiology", "Hemisphere x Gender", "Etiology x Gender"]
print("etiology, stroke:", int((eti > 0).sum()), "tumour resection:", int((eti < 0).sum()))
anc = {}
for cd in ("S1", "B"):
    b = beh[cd][beh[cd].group == 1]
    Y = np.stack([b[b.subject == s].sort_values(["lesion_vhf", "condition"], ascending=[False, True]).acc.to_numpy().reshape(2, 3)
                  for s in PFC])
    anc[cd] = pd.DataFrame(fp.ancova_rm(Y, B, bnames)).set_index("term")
both = anc["S1"][["F", "df1", "df2", "partial_eta2", "p"]].join(anc["B"][["F", "p"]], rsuffix="_codingB")
print(both.round(3).to_string())
for cd in ("S1", "B"):
    print(f"coding {cd}: smallest p = {anc[cd].p.min():.3f} ({anc[cd].p.idxmin()}); terms with p < 0.05: {int((anc[cd].p < 0.05).sum())}")
print("(coding A gives the same F values as S1: it swaps lesion and intact in every patient)")

**Result: reproduced.** No term of the repeated-measures ANCOVA reaches p < 0.05 (smallest p = 0.106, gender, F(1,2) = 7.96), under the Table S1 coding and under coding B; the paper's bound is p > 0.14. The F values do not match Table S2 term by term (for example age F = 7.27 against 4.38, gender 7.96 against 5.33).

### table-s1-patient-accuracy
> Each patient's accuracy in the lesioned and intact visual hemifield is as Table S1 lists,
> from 0.65 to 0.98.

The reference values are Table S1's, typed from the paper (patients 1 to 14, lesion then
intact hemifield); `pfc01` to `pfc14` are taken as patients 1 to 14, whose lesion hemisphere,
etiology, size, age and gender in the docs file match the table row by row (checked below).

In [ ]:
TABLE_S1 = {1: (0.88, 0.85), 2: (0.65, 0.68), 3: (0.98, 0.98), 4: (0.83, 0.85), 5: (0.91, 0.88),
            6: (0.97, 0.94), 7: (0.81, 0.92), 8: (0.97, 0.92), 9: (0.86, 0.96), 10: (0.88, 0.92),
            11: (0.79, 0.70), 12: (0.92, 0.89), 13: (0.92, 0.88), 14: (0.82, 0.80)}
TABLE_S1_ROWS = {1: ("L", 27.44, 48, "F"), 2: ("L", 46.86, 54, "M"), 3: ("L", 0.57, 46, "M"), 4: ("L", 54.94, 34, "M"),
                 5: ("L", 77.61, 64, "F"), 6: ("L", 185.56, 57, "F"), 7: ("L", 11.11, 71, "F"), 8: ("R", 30.81, 22, "F"),
                 9: ("R", 4.74, 20, "F"), 10: ("R", 10.84, 41, "M"), 11: ("R", 56.83, 37, "M"), 12: ("R", 38.09, 29, "F"),
                 13: ("R", 14.87, 62, "F"), 14: ("R", 32.22, 64, "F")}
match_rows = all((pinfo.loc[f"pfc{i:02d}", "lesion_hem"], round(pinfo.loc[f"pfc{i:02d}", "lesion_size"], 2),
                  pinfo.loc[f"pfc{i:02d}", "test_age"], pinfo.loc[f"pfc{i:02d}", "gender"]) == v for i, v in TABLE_S1_ROWS.items())
print("docs rows match Table S1 rows (hemisphere, size, age, gender):", match_rows)
print("docs vhf_lesion/vhf_intact equal Table S1:",
      all((pinfo.loc[f"pfc{i:02d}", "vhf_lesion"], pinfo.loc[f"pfc{i:02d}", "vhf_intact"]) == v for i, v in TABLE_S1.items()))
res_s1 = {}
for cd in ("S1", "A", "B"):
    for src in ("table", "derived"):
        vals = []
        for i in range(1, 15):
            s = f"pfc{i:02d}"
            c = pd.DataFrame(cells(s, cd, src))
            les = c[c.lesion_vhf == 1].k.sum() / c[c.lesion_vhf == 1].n.sum()
            itc = c[c.lesion_vhf == 0].k.sum() / c[c.lesion_vhf == 0].n.sum()
            vals += [(s, "lesion", les, TABLE_S1[i][0]), (s, "intact", itc, TABLE_S1[i][1])]
        v = pd.DataFrame(vals, columns=["subject", "vhf", "measured", "table_s1"])
        v["diff"] = v.measured - v.table_s1
        res_s1[(cd, src)] = v
        print(f"coding {cd:2s}, {src:7s} trials: {int((v['diff'].abs() <= 0.02 + 1e-9).sum())} of 28 within 0.02; "
              f"median |diff| {v['diff'].abs().median():.3f}; measured range {v.measured.min():.2f} to {v.measured.max():.2f}")
v = res_s1[("S1", "table")]
print(v.assign(measured=v.measured.round(3), diff=v["diff"].round(3)).to_string())

**Result: partially reproduced.** 20 of the 28 values are within 0.02 of Table S1 when trial-table hemifield 1 is taken as the lesioned one in every patient (coding S1), with every trial of the trial tables or with the trials kept after cleaning; the 8 misses are the 4 patients recorded in two blocks (pfc10 to pfc13, differences 0.028 to 0.098). The paper's own coding note (the lesioned hemifield is the right one for left-lesioned patients) together with the descriptor's coding of the trial tables (1 left, 2 right) gives 6 to 12 of 28 (codings A and B). Measured values range from 0.65 to 0.98, as in the table.

### accuracy-patients-impaired
> PFC patients are less accurate than controls (mean +- SD 0.87 +- 0.08 against 0.95 +-
> 0.03), a group effect with p < 7 x 10^-5 in the mixed-effects model with group as the only
> fixed effect.

In [ ]:
b = beh["A"].copy()
b["cond_f"] = b.condition.astype(str)
acc_all = {s: R[s]["csv"].correct.mean() for s in CTRL + PFC}
acc_cell = b.groupby("subject").acc.mean()
for nm, a in (("(a) all trials", pd.Series(acc_all)), ("(b) mean of six cells", acc_cell)):
    print(f"{nm}: controls {a[CTRL].mean():.3f} +- {a[CTRL].std():.3f}, patients {a[PFC].mean():.3f} +- {a[PFC].std():.3f}")
print("\nper-group accuracy, by subject (a):")
print("  controls", np.round([acc_all[s] for s in CTRL], 2))
print("  patients", np.round([acc_all[s] for s in PFC], 2))
lm = {}
lm["group only, linear"] = fp.lmm_wald(b, "acc ~ group")
lm["full, linear, condition as number"] = fp.lmm_wald(b, "acc ~ group * lesion_vhf * condition")
lm["full, linear, condition as factor"] = fp.lmm_wald(b, "acc ~ group * lesion_vhf * cond_f")
Xg = np.column_stack([np.ones(len(b)), b.group])
lm["group only, binomial logit"] = fp.glmm_binomial(b.k.to_numpy(float), b.n.to_numpy(float), Xg, b.subject.to_numpy(), ["Intercept", "group"])
Xf = np.column_stack([np.ones(len(b)), b.group, b.lesion_vhf, b.condition, b.group * b.lesion_vhf, b.group * b.condition,
                      b.lesion_vhf * b.condition, b.group * b.lesion_vhf * b.condition]).astype(float)
nf_ = ["Intercept", "group", "lesion_vhf", "condition", "group:lesion_vhf", "group:condition", "lesion_vhf:condition", "group:lesion_vhf:condition"]
lm["full, binomial logit, condition as number"] = fp.glmm_binomial(b.k.to_numpy(float), b.n.to_numpy(float), Xf, b.subject.to_numpy(), nf_)
for k, v in lm.items():
    g = v["group"]
    print(f"{k:45s} group: beta {g['beta']:+.3f}, F(1,{g['df2']}) = {g['F']:.2f}, p = {g['p']:.2e}")
d_cohen = (np.mean([acc_all[s] for s in CTRL]) - np.mean([acc_all[s] for s in PFC])) / np.sqrt(
    (19 * np.var([acc_all[s] for s in CTRL], ddof=1) + 13 * np.var([acc_all[s] for s in PFC], ddof=1)) / 32)
print(f"Cohen's d (pooled SD, subject accuracies) = {d_cohen:.2f}")
print("binomial fits converged:", lm["group only, binomial logit"]["_converged"], lm["full, binomial logit, condition as number"]["_converged"])

**Result: partially reproduced (assumption: the linear model of the Methods).** Mean accuracy is 0.952 +- 0.031 in controls and 0.872 +- 0.099 in patients, within tolerance of 0.95 and 0.87. In the group-only linear mixed model, group F(1,202) = 11.68, p = 7.6e-4, below the Bonferroni level 0.0071 but not below 7e-5; the binomial logit reading gives F = 25.83, p = 8.5e-7. Neither F is within 16.77 +- 1.68. The full model's group effect is F(1,196) = 0.22 (p = 0.64) with condition as a number, against the paper's 4.70 (p = 0.03).

### patients-above-chance
> Despite the impairment, patients perform well above the 0.5 chance rate.

In [ ]:
bp = beh["S1"][beh["S1"].group == 1]
def t_chance(x):
    t = stats.ttest_1samp(x, 0.5, alternative="greater")
    return t.statistic, t.pvalue
t_a = t_chance(np.array([acc_all[s] for s in PFC]))
print(f"(a) overall: t(13) = {t_a[0]:.2f}, one-tailed p = {t_a[1]:.1e}")
for vn, nm in ((1, "lesion"), (0, "intact")):
    x = bp[bp.lesion_vhf == vn].groupby("subject").apply(lambda d: d.k.sum() / d.n.sum(), include_groups=False)
    t = t_chance(x.to_numpy())
    print(f"(b) {nm} hemifield (Table S1 coding): t(13) = {t[0]:.2f}, p = {t[1]:.1e}")
for vn, nm in ((1, "lesion"), (0, "intact")):
    for c in (1, 2, 3):
        x = bp[(bp.lesion_vhf == vn) & (bp.condition == c)].acc.to_numpy()
        t = t_chance(x)
        print(f"(c) {nm}, condition {c}: t(13) = {t[0]:.2f}, p = {t[1]:.1e}, min accuracy {x.min():.2f}")

**Result: reproduced.** On each patient's overall accuracy (reading a), t(13) = 14.13, one-tailed p = 1.4e-9, above 13.89 (15.43 less 10%). Per hemifield (reading b) t = 13.63 and 13.76, and per hemifield and condition (reading c) t = 6.28 to 29.63, so the paper's "t > 15.43" for every test is not met under readings b and c. Every test is far above chance.

### accuracy-no-hemifield-condition-effect
> Accuracy does not differ between visual hemifield presentations or between conditions.

In [ ]:
m3 = {"linear": fp.lmm_wald(b, "acc ~ lesion_vhf * condition")}
X3 = np.column_stack([np.ones(len(b)), b.lesion_vhf, b.condition, b.lesion_vhf * b.condition]).astype(float)
m3["binomial logit"] = fp.glmm_binomial(b.k.to_numpy(float), b.n.to_numpy(float), X3, b.subject.to_numpy(),
                                        ["Intercept", "lesion_vhf", "condition", "lesion_vhf:condition"])
m3["linear, condition as factor"] = fp.lmm_wald(b, "acc ~ lesion_vhf * cond_f")
for k, v in m3.items():
    for term, r_ in v.items():
        if term.startswith("_"):
            continue
        print(f"{k:28s} {term:28s} beta {r_['beta']:+.4f}  F(1,{r_['df2']}) = {r_['F']:.2f}  p = {r_['p']:.3f}")
print("\nvariant, full model (Table S3 first model), coding A:")
for k in ("full, linear, condition as number", "full, binomial logit, condition as number"):
    for term in ("lesion_vhf", "condition", "lesion_vhf:condition"):
        r_ = lm[k][term]
        print(f"  {k:42s} {term:22s} F = {r_['F']:.2f}  p = {r_['p']:.3f}")
bB = beh["B"]
r_B = fp.lmm_wald(bB, "acc ~ lesion_vhf * condition")
print("coding B, linear:", {k: round(v["p"], 3) for k, v in r_B.items()})
print("mean accuracy by hemifield (coding A, subject means of cells):", b.groupby("lesion_vhf").acc.mean().round(3).to_dict(),
      "| by condition:", b.groupby("condition").acc.mean().round(3).to_dict())

**Result: reproduced (assumption: the linear model of the Methods).** In the linear mixed model with hemifield and condition only, hemifield p = 0.915, condition p = 0.094 and their interaction p = 0.715. The binomial logit reading finds a condition effect (F(1,200) = 20.71, p < 0.001), and with condition as a factor the spatial condition differs from identity (p < 0.001); mean accuracy is 0.879, 0.956 and 0.922 in the identity, spatial and temporal conditions. In the full model condition has p = 0.038 (linear).

## EEG: group tests

Every group test below is the cluster test of the paper's scripts (`*_stat_group.m`,
`*_stat_group_hem.m`): independent-samples t of controls minus patients, cluster-forming
threshold p < 0.05 two-tailed (|t(32)| > 2.04), cluster size the sum of t, FieldTrip's
biosemi64 neighbours with minnbchan = 2, 1000 shuffles of the group labels, p doubled
(correcttail 'prob'). A positive cluster means controls above patients. The hemisphere index
is left (lesioned) minus right (intact) homolog, 27 channels. Hemifield h = 2 is the
"lesioned" hemifield of coding A, h = 1 the "intact" one.

In [ ]:
SEED = {"erp": 101, "fft": 202, "pow": 303, "psi": 404, "deg": 505}
def hfor(s, h, coding="A"):
    if coding == "B" and s in PFC and pinfo.loc[s, "lesion_hem"] == "R":
        return 3 - h
    return h
def stack(key, h=None, coding="A", fn=None):
    """Controls and patients stacked; key may contain {h}."""
    def one(s):
        v = R[s][key.format(h=hfor(s, h, coding))] if h is not None else R[s][key]
        return fn(v) if fn else v
    return np.stack([one(s) for s in CTRL]), np.stack([one(s) for s in PFC])
HEM_LABELS = fp.LEFT
def describe(res, chans, axes, n=4, pmax=1.01):
    """Print the n most significant clusters: sign, p, sum of t, channels, and the span of each
    other axis. axes: list of (name, values, unit)."""
    out = []
    for c in res["clusters"][:n]:
        if c["p"] > pmax:
            continue
        m = c["mask"]
        ch = [chans[i] for i in np.flatnonzero(m.reshape(m.shape[0], -1).any(1))]
        spans = []
        for k, (nm, vals, unit) in enumerate(axes):
            other = tuple(i for i in range(m.ndim) if i != k + 1)
            idx = np.flatnonzero(m.any(axis=other))
            spans.append(f"{nm} {vals[idx[0]]:g}-{vals[idx[-1]]:g} {unit}")
        out.append(c)
        print(f"   {'+' if c['sign'] > 0 else '-'} p = {c['p']:.3f}  sum t = {c['stat']:.1f}  {', '.join(spans)}  "
              f"channels ({len(ch)}): {' '.join(ch[:20])}{' ...' if len(ch) > 20 else ''}")
    if not res["clusters"]:
        print("   no cluster")
    return out
def minp(res, sign=None):
    ps = [c["p"] for c in res["clusters"] if sign is None or c["sign"] == sign]
    return min(ps) if ps else 1.0
t_enc_ms = np.arange(385) / fp.FS * 1000
t_proc_ms = np.arange(231) / fp.FS * 1000
TESTS = {}
t_tests = time.time()

### erp-lesioned-maintenance
> For stimuli presented to the lesioned visual hemifield, patients show attenuated positive
> ERPs in posterior channels and reversed-polarity signals in lesioned-hemisphere anterior
> channels early in maintenance.

In [ ]:
POST_S2 = ["P6", "P8", "P10", "PO8"]
ANT_ERP = ["F5", "F7", "FC5", "FT7", "C3", "C5", "T7"]
for coding in ("A", "B"):
    x1, x2 = stack("erp_enc_{h}", 2, coding)
    res = fp.cluster_test(x1, x2, ADJ, seed=SEED["erp"])
    TESTS[("erp_enc_main", 2, coding)] = res
    print(f"coding {coding}: encoding-maintenance, lesioned hemifield, group main effect")
    describe(res, labels, [("time", t_enc_ms, "ms")], n=6)
    for c in res["clusters"]:
        if c["p"] >= 0.05:
            continue
        m = c["mask"]
        tmask = m.any(0)
        late = t_enc_ms[tmask].max() > 400
        post = any(m[L[ch]].any() for ch in POST_S2)
        ant = [ch for ch in ANT_ERP if m[L[ch]].any()]
        mc = x1.mean(0)[m].mean()
        mp = x2.mean(0)[m].mean()
        print(f"   cluster p = {c['p']:.3f}: extends past 400 ms {late}; includes P6/P8/P10/PO8 {post}; "
              f"anterior channels {ant}; mean ERP in cluster controls {mc:.2f}, patients {mp:.2f}")
print("posterior P6-P8-P10-PO8, 400-800 ms, mean ERP (coding A): controls %.2f, patients %.2f" % (
    stack("erp_enc_{h}", 2)[0][:, [L[c] for c in POST_S2]][:, :, (t_enc_ms >= 400) & (t_enc_ms < 800)].mean(),
    stack("erp_enc_{h}", 2)[1][:, [L[c] for c in POST_S2]][:, :, (t_enc_ms >= 400) & (t_enc_ms < 800)].mean()))

**Result: reproduced under coding A (assumption: the descriptor's hemifield coding).** Two clusters pass p < 0.05 after 400 ms: one in F5, F7, FT7, FC5, C3, C5, T7 (with FC3, C1 and others) at 820 to 887 ms, p = 0.006, where the patients' mean ERP is +0.52 and the controls' -0.33 (opposite signs), and one in TP8, P4, P6, P8, P10, PO8 at 754 to 863 ms, p = 0.014, controls above patients (1.23 against -0.09). Both lie in maintenance, about 400 ms after its start; the paper says "early during the maintenance". Under coding B neither cluster survives (smallest p = 0.070).

### erp-processing-interaction
> At active processing, for lesioned-hemifield stimuli, ERPs show a group by hemisphere
> interaction without a group main effect; patients have enhanced signal in
> lesioned-hemisphere anterior channels 100 ms after the test prompt, relative to the intact
> homologs.

The processing segment is aligned to the encoding segment by trial number (the file stores
it in another order). Variant: the selection of `erp_compute.m`, which applies the encoding
segment's trial indices to the processing segment as stored.

In [ ]:
ANT_PROC = ["Fp1", "AF3", "AF7", "F5", "F7", "FC5", "FT7"]
for coding, key in (("A", "erp_proc_{h}"), ("B", "erp_proc_{h}"), ("A", "erp_proc_codeidx_{h}")):
    tag = f"coding {coding}" + (", erp_compute.m trial selection" if "codeidx" in key else "")
    x1, x2 = stack(key, 2, coding)
    rh = fp.cluster_test(fp.hem_index(x1, labels), fp.hem_index(x2, labels), ADJ_HEM, seed=SEED["erp"])
    rm = fp.cluster_test(x1, x2, ADJ, seed=SEED["erp"])
    TESTS[("erp_proc_hem", 2, coding, key)] = rh
    TESTS[("erp_proc_main", 2, coding, key)] = rm
    print(f"{tag}: group x hemisphere")
    describe(rh, HEM_LABELS, [("time", t_proc_ms, "ms")], n=4)
    for c in rh["clusters"]:
        if c["p"] < 0.05:
            m = c["mask"]
            ant = [ch for ch in ANT_PROC if ch in HEM_LABELS and m[HEM_LABELS.index(ch)].any()]
            tm = t_proc_ms[m.any(0)]
            print(f"     p = {c['p']:.3f}: sign {c['sign']} (negative = patients' index above controls'), "
                  f"time {tm.min():.0f}-{tm.max():.0f} ms, anterior channels {ant}")
    print(f"{tag}: group main effect, smallest cluster p = {minp(rm):.3f}")
    describe(rm, labels, [("time", t_proc_ms, "ms")], n=2)
hi1, hi2 = (fp.hem_index(x, labels) for x in stack("erp_proc_{h}", 2))
w100 = (t_proc_ms >= 50) & (t_proc_ms <= 150)
ai = [HEM_LABELS.index(c) for c in ANT_PROC if c in HEM_LABELS]
print("index over FP1-AF3-AF7-F5-F7-FC5-FT7, 50-150 ms (coding A): controls %.2f, patients %.2f" % (
    hi1[:, ai][:, :, w100].mean(), hi2[:, ai][:, :, w100].mean()))

**Result: not reproduced.** No group by hemisphere cluster passes p < 0.05 at active processing (smallest p = 0.294, coding A; 0.500, coding B; 0.272 with the trial selection of `erp_compute.m`), though the patients' index over FP1-AF3-AF7-F5-F7-FC5-FT7 at 50 to 150 ms is higher than the controls' (0.38 against 0.05). There is no group main effect under coding A (smallest p = 0.636); the code's trial selection gives one (p = 0.036, right posterior channels, 55 to 141 ms).

### erp-intact-none
> None of the ERP group effects is seen when stimuli are presented to the intact visual
> hemifield.

In [ ]:
for coding in ("A", "B"):
    x1, x2 = stack("erp_enc_{h}", 1, coding)
    r1 = fp.cluster_test(x1, x2, ADJ, seed=SEED["erp"])
    x1, x2 = stack("erp_proc_{h}", 1, coding)
    r2 = fp.cluster_test(x1, x2, ADJ, seed=SEED["erp"])
    r3 = fp.cluster_test(fp.hem_index(x1, labels), fp.hem_index(x2, labels), ADJ_HEM, seed=SEED["erp"])
    for nm, r_, ch, tm in (("encoding-maintenance main effect", r1, labels, t_enc_ms),
                           ("processing main effect", r2, labels, t_proc_ms),
                           ("processing group x hemisphere", r3, HEM_LABELS, t_proc_ms)):
        TESTS[("erp_intact", nm, coding)] = r_
        print(f"coding {coding}, intact hemifield, {nm}: smallest cluster p = {minp(r_):.3f}")
        describe(r_, ch, [("time", tm, "ms")], n=2)

**Result: reproduced.** For the intact hemifield no test finds a cluster with p < 0.05: encoding-maintenance main effect p = 0.554, processing main effect p = 0.126, processing group by hemisphere p = 0.188 (coding A); 0.518, 0.180 and 0.706 under coding B.

### baseline-spectral-interaction
> In the pretrial baseline, patients have higher raw power between 6 and 18 Hz in
> lesioned-hemisphere anterior channels (AF3, F3, F5) than in the intact homologs, a group by
> hemisphere interaction (a steepened spectral slope over the lesion).

All trials, 500 ms pretrial segment, Hanning taper, zero padded to 7.5 s (`fft_compute.m`),
1 to 40 Hz, raw power averaged over trials.

In [ ]:
f1, f2 = stack("fft")
rf = fp.cluster_test(fp.hem_index(f1, labels), fp.hem_index(f2, labels), ADJ_HEM, seed=SEED["fft"])
TESTS["fft_hem"] = rf
print("baseline power, group x hemisphere:")
describe(rf, HEM_LABELS, [("frequency", fp.FREQS, "Hz")], n=5)
ai = [HEM_LABELS.index(c) for c in ("AF3", "F3", "F5")]
fr = (fp.FREQS >= 6) & (fp.FREQS <= 18)
i1, i2 = fp.hem_index(f1, labels), fp.hem_index(f2, labels)
print("index AF3-F3-F5, 6-18 Hz: controls %.3f, patients %.3f" % (i1[:, ai][:, :, fr].mean(), i2[:, ai][:, :, fr].mean()))
print("per patient:", np.round(i2[:, ai][:, :, fr].mean((1, 2)), 2))

**Result: reproduced.** One group by hemisphere cluster passes, p = 0.004, over AF3, F3, F5 and FC3 from 3 to 19 Hz, with the patients' lesioned-minus-intact power above the controls' (6 to 18 Hz index at AF3-F3-F5: 2.48 against -0.30). The paper reports p = 0.004 over AF3-F3-F5 at 6 to 18 Hz; the cluster here extends down to 3 Hz.

### baseline-no-group-effect
> Baseline raw power shows no main effect of group.

In [ ]:
rf2 = fp.cluster_test(f1, f2, ADJ, seed=SEED["fft"])
TESTS["fft_main"] = rf2
print(f"baseline power, group main effect: smallest cluster p = {minp(rf2):.3f}")
describe(rf2, labels, [("frequency", fp.FREQS, "Hz")], n=4)
tt_ = stats.ttest_ind(f1, f2, axis=0)
print(f"channel-frequency points with uncorrected p < 0.05: {int((tt_.pvalue < 0.05).sum())} of {tt_.pvalue.size}")
pl8 = [L[c] for c in ("P1", "P3", "P5", "P7", "P9", "PO3", "PO7", "O1")]
a8 = (fp.FREQS >= 8) & (fp.FREQS <= 12)
print("posterior lesioned-side alpha (8-12 Hz) power: controls %.2f, patients %.2f" % (f1[:, pl8][:, :, a8].mean(), f2[:, pl8][:, :, a8].mean()))

**Result: partially reproduced.** No cluster of the group main effect passes (smallest p = 0.066, controls above patients at 13 to 40 Hz over 23 anterior channels), but 476 of 2560 channel-frequency points have uncorrected p < 0.05, where the paper states uncorrected p > 0.05; the criterion marks the claim as only partly holding when any point passes.

## Task-induced power

Readings run for every power claim (data check: the segments have no buffers):
- window **pad**: the 3/f windows run into the zero padding, so every frequency has values
  (the assumption adopted; it is the only reading under which the 3-4 Hz results of the paper
  exist); **fit**: FieldTrip's rule, NaN wherever the window leaves the segment, which leaves no
  baseline below 7 Hz and nothing at 3-4 Hz.
- r of the bootstrap: **hemifield** (`zbaseline.m`, the trials being scored) or **all** (the
  Methods' "number of trials in that subject's dataset").

In [ ]:
t_enc_s = R[CTRL[0]]["tfr_enc_time"]
t_proc_s = R[CTRL[0]]["tfr_proc_time"]
def zstack(seg, h, reading="pad", r_mode="hemifield", coding="A"):
    """Power Z of controls and patients, (subjects, 64, 40, time)."""
    return (np.stack([fp.power_z(R[s], seg, hfor(s, h, coding), reading, r_mode) for s in CTRL]),
            np.stack([fp.power_z(R[s], seg, hfor(s, h, coding), reading, r_mode) for s in PFC]))
def band_course(z, chans, fmin, fmax):
    """Mean over channels and frequencies (NaN-aware) -> (subjects, time)."""
    fi = (fp.FREQS >= fmin) & (fp.FREQS <= fmax)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        return np.nanmean(z[:, [L[c] for c in chans]][:, :, fi], axis=(1, 2))
ANT_S4 = ["AF7", "F5", "F7", "FC5", "FT7"]
POST_S4 = ["P1", "P3", "P5", "P7", "P9", "PO3", "PO7", "O1"]

### power-anterior-theta-controls
> In controls, delta-theta (2 to 7 Hz) power in anterior channels rises during
> encoding-maintenance and stays elevated through active processing, where it peaks, above
> Z = 3.29 against the pretrial baseline.

In [ ]:
rows = []
for reading in ("pad", "fit"):
    for r_mode in ("hemifield", "all"):
        ce = {h: band_course(zstack("enc", h, reading, r_mode)[0], ANT_S4, 2, 7) for h in (1, 2)}
        cp = {h: band_course(zstack("proc", h, reading, r_mode)[0], ANT_S4, 2, 7) for h in (1, 2)}
        for hf in ("pooled", 1, 2):
            e = (ce[1] + ce[2]) / 2 if hf == "pooled" else ce[hf]
            p = (cp[1] + cp[2]) / 2 if hf == "pooled" else cp[hf]
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                ge, gp = np.nanmean(e, 0), np.nanmean(p, 0)
            full = np.concatenate([ge, gp])
            peak_in_proc = bool(np.nanargmax(full) >= len(ge)) if np.isfinite(full).any() else False
            maj_p = np.nanmean(p > 3.29, 0) if np.isfinite(p).any() else np.array([np.nan])
            rows.append({"window": reading, "r": r_mode, "hemifield": hf,
                         "enc mean Z 0-0.4 s": np.nanmean(ge[t_enc_s < 0.4]) if np.isfinite(ge).any() else np.nan,
                         "enc mean Z 0.4-1.5 s": np.nanmean(ge[t_enc_s >= 0.4]) if np.isfinite(ge).any() else np.nan,
                         "proc max group-mean Z": np.nanmax(gp) if np.isfinite(gp).any() else np.nan,
                         "proc frac. time group mean > 3.29": np.nanmean(gp > 3.29) if np.isfinite(gp).any() else np.nan,
                         "max fraction of controls > 3.29 (proc)": np.nanmax(maj_p),
                         "group-mean max lies in proc": peak_in_proc,
                         "time of max (s, proc if in proc)": (t_proc_s[np.nanargmax(gp)] if peak_in_proc else t_enc_s[np.nanargmax(ge)]) if np.isfinite(full).any() else np.nan})
tab = pd.DataFrame(rows)
with pd.option_context("display.width", 250, "display.max_columns", 20):
    print(tab.round(2).to_string())
e = (band_course(zstack("enc", 1)[0], ANT_S4, 2, 7) + band_course(zstack("enc", 2)[0], ANT_S4, 2, 7)) / 2
p = (band_course(zstack("proc", 1)[0], ANT_S4, 2, 7) + band_course(zstack("proc", 2)[0], ANT_S4, 2, 7)) / 2
print("\nper control (pad, r hemifield, pooled): max Z in processing", np.round(np.nanmax(p, 1), 1))
print("                                         max Z in encoding-maintenance", np.round(np.nanmax(e, 1), 1))
print("group mean course, encoding-maintenance every 100 ms:", np.round(e.mean(0)[::10], 1))
print("group mean course, processing every 100 ms:", np.round(p.mean(0)[::10], 1))
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(t_enc_s, e.mean(0), label="encoding-maintenance")
ax.plot(1.5 + 0.1 + t_proc_s, p.mean(0), label="processing (shifted by 1.6 s)")
ax.axhline(3.29, ls=":", c="k"); ax.set_xlabel("s"); ax.set_ylabel("Z, 2-7 Hz, anterior"); ax.legend(fontsize=7)
ax.set_title("controls, pad, r hemifield, hemifields pooled"); plt.show()
print("plotted range:", round(float(np.nanmin([e.mean(0).min(), p.mean(0).min()])), 1), "to", round(float(np.nanmax([e.mean(0).max(), p.mean(0).max()])), 1))

**Result: reproduced (assumption: windows may run into the zero padding).** With the padded windows, the controls' 2 to 7 Hz Z over AF7-F5-F7-FC5-FT7 rises during encoding-maintenance (group mean 2.2 over 0 to 0.4 s, 1.8 later), exceeds 3.29 during active processing (maximum 3.74 at 0.53 s, hemifields pooled; 3.81 and 3.69 per hemifield) and peaks there; with r = all correct trials the maximum is 5.32. The proportion of controls above 3.29 at any processing time is at most 0.55. Under FieldTrip's window rule only 7 Hz of the band has a baseline and the maximum is 0.81 to 1.36, so the claim does not hold under that reading.

### power-posterior-alphabeta
> Posterior channels show beta-gamma (12 to 35 Hz) increases and alpha desynchronization at
> encoding, then alpha-beta (9 to 24 Hz) decreases through maintenance (|Z| > 1.96) and active
> processing (Z < -3.29), in both groups.

In [ ]:
rows = []
for reading in ("pad", "fit"):
    for r_mode in ("hemifield", "all"):
        for gi, gname in ((0, "controls"), (1, "patients")):
            ze = {h: zstack("enc", h, reading, r_mode)[gi] for h in (1, 2)}
            zp = {h: zstack("proc", h, reading, r_mode)[gi] for h in (1, 2)}
            for hf in ("pooled", 1, 2):
                pick = (lambda d: (d[1] + d[2]) / 2) if hf == "pooled" else (lambda d, hf=hf: d[hf])
                ab_e = np.nanmean(band_course(pick(ze), POST_S4, 9, 24), 0)
                ab_p = np.nanmean(band_course(pick(zp), POST_S4, 9, 24), 0)
                bg_e = np.nanmean(band_course(pick(ze), POST_S4, 12, 35), 0)
                maint = (t_enc_s >= 0.4)
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", RuntimeWarning)
                    rows.append({"window": reading, "r": r_mode, "group": gname, "hemifield": hf,
                                 "maint frac < -1.96": np.nanmean(ab_e[maint] < -1.96), "maint max Z": np.nanmax(ab_e[maint]),
                                 "proc frac < -3.29": np.nanmean(ab_p < -3.29), "proc max Z": np.nanmax(ab_p),
                                 "12-35 Hz mean Z 0-0.4 s": np.nanmean(bg_e[t_enc_s < 0.4])})
tab2 = pd.DataFrame(rows)
with pd.option_context("display.width", 250, "display.max_columns", 20, "display.max_rows", 60):
    print(tab2.round(2).to_string())
zc = zstack("enc", 2)
print("\nalpha (8-12 Hz) posterior, 0-0.4 s, lesioned hemifield, pad: controls %.2f, patients %.2f" % (
    np.nanmean(band_course(zc[0], POST_S4, 8, 12)[:, t_enc_s < 0.4]), np.nanmean(band_course(zc[1], POST_S4, 8, 12)[:, t_enc_s < 0.4])))

**Result: not reproduced.** The posterior 9 to 24 Hz Z is below -1.96 at no more than 4% of maintenance time points with the padded windows, and at most 20% under FieldTrip's rule (r = all trials), in either group. In processing it is below -3.29 at 0 to 28% (r = trials of the hemifield) and up to 54% (r = all trials) of time points. The 12 to 35 Hz Z is above zero at encoding in both groups (0.38 to 1.10), and 8 to 12 Hz is slightly negative at 0 to 0.4 s (-0.42 controls, -0.51 patients).

### power-lowtheta-interaction
> At active processing, patients have reduced low-theta (3 to 4 Hz) power in
> lesioned-hemisphere anterior channels relative to the intact homologs, a group by
> hemisphere interaction found both when stimuli went to the lesioned and to the intact visual
> hemifield.

O2 readings: (a) channels by frequency (1 to 40 Hz) by time, as `tfr_stat_group_hem.m` runs
it; (b) channels by time on the 3-4 Hz mean. Window reading pad (fit has no value at 3-4 Hz).
A positive cluster is the patients' index below the controls'.

In [ ]:
ANT_LES = ["Fp1", "AF7", "AF3", "F1", "F3", "F5", "F7", "FT7", "FC5", "FC3", "FC1"]
lowtheta = {}
f34 = (fp.FREQS >= 3) & (fp.FREQS <= 4)
for h in (2, 1):
    z1, z2 = zstack("proc", h)
    i1, i2 = fp.hem_index(z1, labels), fp.hem_index(z2, labels)
    ra = fp.cluster_test(i1, i2, ADJ_HEM, seed=SEED["pow"])
    rb = fp.cluster_test(i1[:, :, f34].mean(2), i2[:, :, f34].mean(2), ADJ_HEM, seed=SEED["pow"])
    TESTS[("pow_hem", h, "a")] = ra
    TESTS[("pow_hem", h, "b")] = rb
    print(f"hemifield {h} ({'lesioned' if h == 2 else 'intact'}), (a) channel x frequency x time:")
    describe(ra, HEM_LABELS, [("frequency", fp.FREQS, "Hz"), ("time", t_proc_s * 1000, "ms")], n=4)
    for c in ra["clusters"]:
        if c["p"] < 0.05:
            m = c["mask"]
            cov34 = bool(m[:, f34].any())
            ant = [HEM_LABELS[i] for i in np.flatnonzero(m[:, f34].any((1, 2)))] if cov34 else []
            print(f"     p = {c['p']:.3f}, sign {c['sign']}: covers 3-4 Hz {cov34}; anterior channels at 3-4 Hz "
                  f"{[a for a in ant if a in ANT_LES]}")
    print(f"hemifield {h}, (b) channel x time, 3-4 Hz:")
    describe(rb, HEM_LABELS, [("time", t_proc_s * 1000, "ms")], n=3)
    for c in rb["clusters"]:
        if c["p"] < 0.05:
            lowtheta.setdefault(h, []).append([HEM_LABELS[i] for i in np.flatnonzero(c["mask"].any(1))])
    ai = [HEM_LABELS.index(c) for c in ANT_S4]
    print("   index 3-4 Hz, AF7-F5-F7-FC5-FT7, whole segment: controls %.2f, patients %.2f" % (
        np.nanmean(i1[:, ai][:, :, f34]), np.nanmean(i2[:, ai][:, :, f34])))
print("significant 3-4 Hz clusters (reading b), channels:", lowtheta)

**Result: partially reproduced (assumptions: padded windows, and the test on the 3-4 Hz mean).** On the 3 to 4 Hz mean (reading b) the lesioned hemifield gives a group by hemisphere cluster over AF7, F5, F7, FT7 and FC5 at 12 to 352 ms, p = 0.040, with the patients' index below the controls' (-1.79 against 0.27 over the segment); the intact hemifield's cluster in F5-F7-FT7-FC5 has p = 0.134. On channels by frequency by time (reading a, as `tfr_stat_group_hem.m` runs it) neither hemifield has a significant cluster (p = 0.508 and 0.834), though the leading cluster lies at 3 to 5 Hz in the same five channels.

### power-no-group-main-effect
> Task-induced power shows no group main effect, neither for anterior theta nor for the
> posterior alpha-beta decreases.

In [ ]:
nomain = []
for seg, tt_s in (("enc", t_enc_s), ("proc", t_proc_s)):
    for h in (2, 1):
        for reading in ("pad", "fit"):
            z1, z2 = zstack(seg, h, reading)
            r_ = fp.cluster_test(z1, z2, ADJ, seed=SEED["pow"])
            TESTS[("pow_main", seg, h, reading)] = r_
            nomain.append({"segment": seg, "hemifield": h, "reading": f"(a) {reading}", "min cluster p": minp(r_)})
            print(f"{seg}, hemifield {h}, window {reading}, chan x freq x time: smallest p = {minp(r_):.3f}")
            describe(r_, labels, [("frequency", fp.FREQS, "Hz"), ("time", tt_s * 1000, "ms")], n=2, pmax=0.2)
        z1, z2 = zstack(seg, h, "pad")
        for fmin, fmax in ((3, 4), (2, 7), (9, 24)):
            fi = (fp.FREQS >= fmin) & (fp.FREQS <= fmax)
            r_ = fp.cluster_test(z1[:, :, fi].mean(2), z2[:, :, fi].mean(2), ADJ, seed=SEED["pow"])
            nomain.append({"segment": seg, "hemifield": h, "reading": f"(b) {fmin}-{fmax} Hz", "min cluster p": minp(r_)})
        g_ant = band_course(z1, ANT_S4, 3, 4).mean() - band_course(z2, ANT_S4, 3, 4).mean()
        g_post = band_course(z1, POST_S4, 9, 24).mean() - band_course(z2, POST_S4, 9, 24).mean()
        print(f"   controls minus patients, anterior 3-4 Hz Z {g_ant:+.2f}, posterior 9-24 Hz Z {g_post:+.2f}")
print(pd.DataFrame(nomain).round(3).to_string())

**Result: reproduced.** No group main effect cluster passes on channels by frequency by time, in either segment, hemifield or window reading (smallest p = 0.388). On band means (reading b) the smallest p is 0.062 (3 to 4 Hz, encoding-maintenance). Controls minus patients is +1.2 to +1.3 Z in anterior 3 to 4 Hz during encoding-maintenance and about 0 in processing; posterior 9 to 24 Hz differs by -0.43 to +0.14.

## Directional connectivity (PSI)

PSI Z per subject is computed in the helper under three readings (see its docstring):
**trialwise** (adopted: the structure of `psianalysis.m`, per-trial phase slopes averaged
over trials and Z scored on the frequency-shuffled trial-mean coherency, with the complex
conjugate the Methods' "cross-spectral density" implies), **nolte** (PSI of the trial-averaged
coherency, Nolte et al. 2008, null from one shuffle of the bins) and **literal** (`psianalysis.m`
exactly as written, without the conjugate). The window reading is pad: under fit no
delta-theta value exists at all (data check), and alpha-beta exists only at the points listed
there. Z is baseline corrected by the temporal mean of the pretrial Z (`psi_compute.m`).
Positive Z(A, B): A leads B.

The literal code's normalised cross-spectrum does not depend on channel A's phase except for
its sign; on random coefficients:

In [ ]:
rng_lit = np.random.default_rng(7)
xa = rng_lit.standard_normal(6) + 1j * rng_lit.standard_normal(6)
xb = rng_lit.standard_normal(6) + 1j * rng_lit.standard_normal(6)
pp1 = fp.psianalysis_literal_pp(xa, xb)
pp2 = fp.psianalysis_literal_pp(xa * np.exp(1j * rng_lit.uniform(0, 2 * np.pi, 6)), xb)
print("literal pp, |pp| =", np.round(np.abs(pp1), 3))
print("pp divided by exp(2i phase(B)):", np.round(pp1 / np.exp(2j * np.angle(xb)), 3))
print("after changing A's phases at random:", np.round(pp2 / np.exp(2j * np.angle(xb)), 3))
t_psi_enc = R[CTRL[0]]["psi_dt_enc_time"]
t_psi_proc = R[CTRL[0]]["psi_dt_proc_time"]
print("PSI times (s): encoding-maintenance", np.round(t_psi_enc, 2), "processing", np.round(t_psi_proc, 2))

In [ ]:
PFC_ROI = {"(b) Figure S4 ANTERIOR": ANT_S4}
if lowtheta.get(2):
    PFC_ROI["(a) power-lowtheta cluster"] = [c for c in lowtheta[2][0] if c in ANT_LES]
print("PFC regions of interest:", PFC_ROI)
PO_ROI = POST_S4
def roi_course(band, h, reading, roi, window="pad"):
    """Baseline-corrected PSI Z from the PFC region to the parieto-occipital region, mean over
    pairs: (controls, time), (patients, time) for encoding-maintenance then processing."""
    a = [L[c] for c in roi]
    b = [L[c] for c in PO_ROI]
    def one(s):
        e = fp.psi_bc(R[s], band, "enc", h, reading, window)[np.ix_(a, b)].mean((0, 1))
        p = fp.psi_bc(R[s], band, "proc", h, reading, window)[np.ix_(a, b)].mean((0, 1))
        return e, p
    ce = [one(s) for s in CTRL]
    pe = [one(s) for s in PFC]
    return (np.stack([x[0] for x in ce]), np.stack([x[1] for x in ce])), (np.stack([x[0] for x in pe]), np.stack([x[1] for x in pe]))
COURSES = {}
for band in ("dt", "ab"):
    for reading in ("trialwise", "nolte", "literal"):
        for rn, roi in PFC_ROI.items():
            for h in (2, 1):
                COURSES[(band, reading, rn, h)] = roi_course(band, h, reading, roi)
def show_course(band, reading, rn, h):
    (ce, cp), (pe, pp_) = COURSES[(band, reading, rn, h)]
    print(f"  {band} {reading:9s} {rn} h{h}: controls enc {np.round(ce.mean(0), 1)} proc {np.round(cp.mean(0), 1)}")
    print(f"  {'':2s} {'':9s} {'':{len(rn)}s}    patients enc {np.round(pe.mean(0), 1)} proc {np.round(pp_.mean(0), 1)}")

### psi-deltatheta-bottomup-encoding
> In both groups, delta-theta PSI shows parieto-occipital regions leading PFC at the offset of
> the second stimulus at encoding, at baseline-corrected Z of -10 or below.

In [ ]:
near400 = (t_psi_enc >= 0.3 - 1e-6) & (t_psi_enc <= 0.5 + 1e-6)
rows = []
for reading in ("trialwise", "nolte", "literal"):
    for rn in PFC_ROI:
        for h in (2, 1):
            (ce, _), (pe, _) = COURSES[("dt", reading, rn, h)]
            for gname, x in (("controls", ce), ("patients", pe)):
                rows.append({"reading": reading, "ROI": rn, "hemifield": h, "group": gname,
                             "min group-mean Z 300-500 ms": x.mean(0)[near400].min(),
                             "fraction of subjects <= -9 at some point there": np.mean((x[:, near400] <= -9).any(1)),
                             "fraction <= -1.96": np.mean((x[:, near400] <= -1.96).any(1))})
print(pd.DataFrame(rows).round(2).to_string())
for reading in ("trialwise", "nolte", "literal"):
    show_course("dt", reading, list(PFC_ROI)[0], 2)

**Result: not reproduced.** No reading gives the PFC to parieto-occipital delta-theta Z at -9 or below near 400 ms in both groups. With the adopted trialwise PSI the group means at 300 to 500 ms stay positive (minima 4.4 to 8.6); with Nolte's PSI they lie between -0.21 and 0.06; with the literal code the controls' are positive (7.6 to 12.3) and the patients' -6.2 to -6.6.

### psi-deltatheta-pfc-lead-controls
> In delta-theta PSI, controls shift to PFC leading parieto-occipital regions in
> mid-maintenance (Z > 1.96), and the PFC lead peaks early in active processing (Z of 10 or
> more), whereas patients show zero directionality (Z < 1.96).

In [ ]:
maint = t_psi_enc >= 0.4 - 1e-6
early = t_psi_proc <= 0.45
rows = []
for reading in ("trialwise", "nolte", "literal"):
    for rn in PFC_ROI:
        for h in (2, 1):
            (ce, cp), (pe, pp_) = COURSES[("dt", reading, rn, h)]
            gc = np.concatenate([ce.mean(0), cp.mean(0)])
            gpat = np.concatenate([pe.mean(0)[maint], pp_.mean(0)])
            rows.append({"reading": reading, "ROI": rn, "hemifield": h,
                         "ctrl max Z maintenance": ce.mean(0)[maint].max(),
                         "ctrl max Z early proc": cp.mean(0)[early].max(),
                         "ctrl max lies in early proc": bool(np.argmax(gc) >= len(ce.mean(0)) and np.argmax(gc) - len(ce.mean(0)) < early.sum()),
                         "patients max Z (maint+proc)": gpat.max(),
                         "ctrl majority > 9 early proc": bool((np.mean(cp[:, early] >= 9, 0) > 0.5).any()),
                         "patients majority < 1.96 throughout": bool((np.mean(np.concatenate([pe[:, maint], pp_], 1) < 1.96, 0) > 0.5).all())})
print(pd.DataFrame(rows).round(2).to_string())

**Result: not reproduced.** In no reading does the controls' PFC lead peak early in processing: with the trialwise PSI their maximum in early processing is 4.3 and 0.2 against 19.2 and 15.0 in maintenance, and the patients reach 14.4; Nolte's PSI stays within +-0.4; the literal code gives controls 21 to 22 in early processing but larger values at the segment edges (up to 227), and patients above 1.96.

### psi-alphabeta-bottomup
> In alpha-beta (9 to 24 Hz) PSI, parieto-occipital regions lead PFC throughout encoding,
> maintenance and active processing, for both visual hemifields.

In [ ]:
rows = []
for reading in ("trialwise", "nolte", "literal"):
    for rn in PFC_ROI:
        for h in (2, 1):
            (ce, cp), (pe, pp_) = COURSES[("ab", reading, rn, h)]
            for gname, e, p in (("controls", ce, cp), ("patients", pe, pp_)):
                rows.append({"reading": reading, "ROI": rn, "hemifield": h, "group": gname,
                             "enc frac < -1.96": np.mean(e.mean(0) < -1.96), "proc frac < -1.96": np.mean(p.mean(0) < -1.96),
                             "enc frac < 0": np.mean(e.mean(0) < 0), "proc frac < 0": np.mean(p.mean(0) < 0),
                             "median Z": np.median(np.concatenate([e.mean(0), p.mean(0)]))})
print(pd.DataFrame(rows).round(2).to_string())
for reading in ("trialwise", "nolte", "literal"):
    show_course("ab", reading, list(PFC_ROI)[0], 2)
# window reading fit, alpha-beta, where values exist
for reading in ("trialwise", "literal"):
    a = [L[c] for c in ANT_S4]; b_ = [L[c] for c in PO_ROI]
    for h in (2, 1):
        v = np.stack([np.concatenate([fp.psi_bc(R[s], "ab", "enc", h, reading, "fit")[np.ix_(a, b_)].mean((0, 1)),
                                      fp.psi_bc(R[s], "ab", "proc", h, reading, "fit")[np.ix_(a, b_)].mean((0, 1))]) for s in CTRL + PFC])
        g = np.nanmean(v[:len(CTRL)], 0)
        print(f"fit window, {reading}, h{h}: controls' group-mean Z where defined {np.round(g[np.isfinite(g)], 1)}")

**Result: not reproduced (assumption: PSI with the conjugate, as the Methods' cross-spectral density implies).** With the trialwise PSI the controls' alpha-beta Z from PFC to parieto-occipital channels is mostly positive (median 3.7 and 3.8; below -1.96 at 11 to 25% of time points). Nolte's PSI never passes -1.96. The literal code of `psianalysis.m` reproduces the claim: Z below -1.96 at 94% of encoding-maintenance and 100% of processing time points in both groups and hemifields (median about -23).

### psi-deltatheta-group-clusters
> Across all 64 by 64 channel pairs, delta-theta PSI differs between groups intermittently
> during encoding and maintenance and in a sustained way during active processing.

Clustering readings: **(c)** per seed channel A, over neighbours of B and time, each seed its
own test (`psi_stat_group.m`; adopted, as the paper's code settles it); (a) per pair over
time; (b) over time and pairs whose both channels are equal or neighbours.

In [ ]:
PAIRS, PADJ = fp.pair_adjacency(ADJ, "pairs")
_, SADJ = fp.pair_adjacency(ADJ, "seed")
CLUST = {"(c) per seed": (SADJ, PAIRS[:, 0], 2), "(a) per pair": (None, None, 0), "(b) pairs": (PADJ, None, 2)}
def psi_pairs(band, seg, h, reading):
    def one(s):
        v = fp.psi_bc(R[s], band, seg, h, reading)
        return v[PAIRS[:, 0], PAIRS[:, 1]]
    return np.stack([one(s) for s in CTRL]), np.stack([one(s) for s in PFC])
def coverage(res, times, pthr=0.05):
    """Times (ms) covered by any cluster with p < pthr, by sign."""
    out = {}
    for c in res["clusters"]:
        if c["p"] < pthr:
            ti = np.flatnonzero(c["mask"].any(0))
            out.setdefault(c["sign"], set()).update(np.round(times[ti] * 1000).astype(int).tolist())
    return {k: sorted(v) for k, v in out.items()}
PSI_TESTS = {}
def run_psi(band, readings_clust, psi_readings):
    rows = []
    for reading in psi_readings:
        for seg, tms in (("enc", t_psi_enc), ("proc", t_psi_proc)):
            for h in (2, 1):
                x1, x2 = psi_pairs(band, seg, h, reading)
                for cn in readings_clust:
                    adj, grp, mn = CLUST[cn]
                    r_ = fp.cluster_test(x1, x2, adj, seed=SEED["psi"], minnbchan=mn, node_groups=grp)
                    PSI_TESTS[(band, reading, seg, h, cn)] = r_
                    cov = coverage(r_, tms)
                    nsig = sum(c["p"] < 0.05 for c in r_["clusters"])
                    rows.append({"PSI": reading, "segment": seg, "hemifield": h, "clustering": cn,
                                 "min p": round(minp(r_), 3), "significant clusters": nsig,
                                 "ms covered, controls > patients": cov.get(1, []),
                                 "ms covered, patients > controls": cov.get(-1, [])})
    return pd.DataFrame(rows)

In [ ]:
t_psi_tests = time.time()
dt_tab = pd.concat([run_psi("dt", list(CLUST), ["trialwise"]), run_psi("dt", ["(c) per seed"], ["nolte", "literal"])])
with pd.option_context("display.width", 300, "display.max_colwidth", 80, "display.max_rows", 60):
    print(dt_tab.to_string())
print("windows of the paper: lesioned (h2) enc 100-200 and 700-800 ms, proc 100-700 ms; intact (h1) enc 200-300 and 1000-1200 ms, proc 0-500 ms")
print(f"PSI tests took {(time.time() - t_psi_tests) / 60:.1f} min")

**Result: not reproduced.** With the adopted readings (trialwise PSI, per-seed clustering of `psi_stat_group.m`) significant clusters cover 2 of the 7 processing time points of the lesioned window (100 and 400 ms of 100 to 700 ms) and 2 of 6 of the intact window (0 and 100 ms), not most of either; the encoding-maintenance windows are touched in the lesioned hemifield (200 ms, and 700 to 800 ms) and in the intact one only at 700 and 800 ms, outside its windows (200 to 300 and 1000 to 1200 ms). Clustering per pair or over neighbouring pairs finds no cluster (smallest p = 0.56). The literal code covers 4 of 7 lesioned processing points (100 to 500 ms) but 2 of 6 intact ones.

### psi-network-degree
> A post hoc network analysis of delta-theta PSI shows group differences from lesioned PFC to
> bilateral central-posterior regions, sustained for 200 ms early in active processing, for
> both visual hemifields.

Degree on the Z scored PSI before baseline correction, then minus the temporal mean of the
baseline degree (`psi_compute.m`). Readings: **ft_count** (`ft_networkanalysis` 'degrees' on
Z > 1.96: in- plus out-degree; the code's), (a) out_signed, (b) out_count, (c) abs_both.
Group test over channels by time (`psi_degree_stat_group.m`).

In [ ]:
rows = []
for reading in ("trialwise", "literal"):
    for mode in ("ft_count", "out_signed", "out_count", "abs_both"):
        for h in (2, 1):
            def deg(s):
                base = fp.degree(fp.psi_raw(R[s], "dt", "pre", None, reading), mode).mean(-1)
                return fp.degree(fp.psi_raw(R[s], "dt", "proc", h, reading), mode) - base[:, None]
            d1 = np.stack([deg(s) for s in CTRL]); d2 = np.stack([deg(s) for s in PFC])
            r_ = fp.cluster_test(d1, d2, ADJ, seed=SEED["deg"])
            TESTS[("deg", reading, mode, h)] = r_
            for c in r_["clusters"][:3]:
                if c["p"] >= 0.1:
                    continue
                ti = np.flatnonzero(c["mask"].any(0))
                ch = [labels[i] for i in np.flatnonzero(c["mask"].any(1))]
                rows.append({"PSI": reading, "degree": mode, "hemifield": h, "p": round(c["p"], 3),
                             "sign (+ controls > patients)": c["sign"], "ms": (t_psi_proc[ti] * 1000).round().astype(int).tolist(),
                             "channels": " ".join(ch)})
            if not any(c["p"] < 0.1 for c in r_["clusters"]):
                rows.append({"PSI": reading, "degree": mode, "hemifield": h, "p": round(minp(r_), 3),
                             "sign (+ controls > patients)": None, "ms": [], "channels": ""})
with pd.option_context("display.width", 300, "display.max_colwidth", 120, "display.max_rows", 60):
    print(pd.DataFrame(rows).to_string())

**Result: not reproduced.** No reading gives a cluster with p <= 0.01 of patients below controls early in processing in both hemifields. The closest are, with the trialwise PSI, a cluster of the sum of |Z| at 0 to 100 ms in right central-posterior channels (FC6 to PO8), p = 0.046, lesioned hemifield only, and the code's degree count at 800 ms (p = 0.022); with the literal code the out-degree count gives p = 0.006 at 0 to 100 ms but with patients above controls.

### psi-alphabeta-no-group-effect
> PFC lesions do not affect alpha-beta PSI at any point of encoding, maintenance or active
> processing.

In [ ]:
ab_tab = pd.concat([run_psi("ab", list(CLUST), ["trialwise"]), run_psi("ab", ["(c) per seed"], ["nolte", "literal"])])
with pd.option_context("display.width", 300, "display.max_colwidth", 80, "display.max_rows", 60):
    print(ab_tab.to_string())
for reading in ("trialwise", "literal"):
    for h in (2, 1):
        (ce, cp), (pe, pp_) = COURSES[("ab", reading, list(PFC_ROI)[0], h)]
        print(f"{reading}, h{h}: controls minus patients, PFC to parieto-occipital Z, enc {np.mean(ce) - np.mean(pe):+.2f}, proc {np.mean(cp) - np.mean(pp_):+.2f}")
print(f"all group tests took {(time.time() - t_tests) / 60:.1f} min; whole notebook so far {(time.time() - T0) / 60:.1f} min; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

**Result: not reproduced (assumptions: trialwise PSI, per-seed clustering).** The per-seed test of `psi_stat_group.m` finds alpha-beta clusters with p < 0.05 in three of four segment and hemifield combinations (smallest p = 0.010); it tests each of the 64 seed channels at 0.05 with no correction across seeds, and Nolte's PSI and the literal code also give clusters (smallest p 0.002 and 0.020). Clustering per pair or over neighbouring pairs finds none (p = 1.0). Controls minus patients in the PFC to parieto-occipital Z is +3.3 to +6.9 with the trialwise PSI.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| claim | outcome | rests on an assumption |
|---|---|---|
| cohort-and-trials | reproduced | no |
| demographics-matched | reproduced | no |
| patient-factors-no-effect | reproduced | no |
| table-s1-patient-accuracy | partially reproduced | no |
| accuracy-patients-impaired | partially reproduced | yes (linear model) |
| patients-above-chance | reproduced | no |
| accuracy-no-hemifield-condition-effect | reproduced | yes (linear model) |
| erp-lesioned-maintenance | reproduced | yes (hemifield coding A) |
| erp-processing-interaction | not reproduced | no |
| erp-intact-none | reproduced | no |
| baseline-spectral-interaction | reproduced | no |
| baseline-no-group-effect | partially reproduced | no |
| power-anterior-theta-controls | reproduced | yes (padded windows) |
| power-posterior-alphabeta | not reproduced | no |
| power-lowtheta-interaction | partially reproduced | yes (padded windows, 3-4 Hz mean) |
| power-no-group-main-effect | reproduced | no |
| psi-deltatheta-bottomup-encoding | not reproduced | no |
| psi-deltatheta-pfc-lead-controls | not reproduced | no |
| psi-deltatheta-group-clusters | not reproduced | no |
| psi-network-degree | not reproduced | no |
| psi-alphabeta-bottomup | not reproduced | yes (PSI with the conjugate) |
| psi-alphabeta-no-group-effect | not reproduced | yes (PSI reading, per-seed test) |

**How the analysis differs from the paper's.** It runs in Python, not MATLAB and FieldTrip: the
spectral estimates, the cluster test (with minnbchan and FieldTrip's neighbour template) and the
models are reimplemented. The bootstrap of the power Z is replaced by its exact expectation.
Three choices the paper does not settle change outcomes and are stated as assumptions: the
hemifield coding of `data_derived.mat` (the descriptor's, A), letting the 3/f windows run into
the zero padding (the segments have no buffers, so FieldTrip's rule leaves no value below 7 Hz in
the baseline), and the PSI (per-trial phase slopes as `psianalysis.m` computes them, but with the
complex conjugate of a cross-spectral density). The processing segment is matched to the trials by
trial number.

**What was found about the dataset.**
- The processing segments of `data_derived.mat` are stored in another trial order than the
  pretrial and encoding-maintenance segments in all 34 subjects; `erp_compute.m` and
  `tfr_compute.m` select processing trials with the encoding segment's indices, so they average
  trials of the wrong accuracy and hemifield.
- The derived hemifield column, described as 1 intact and 2 lesioned, equals the trial tables'
  left/right column in every trial of every subject, for left- and right-lesioned patients alike.
  The docs summary file (and Table S1) give the patients' lesioned-hemifield accuracy as that of
  hemifield 1 in the 9 patients with one trial table, which contradicts the paper's own note that
  the lesioned hemifield is the right one for left-lesioned patients. Which 10 controls were
  swapped is not recorded.
- The segments have no time buffers (500, 1500 and 900 ms), so the deposited `tfr_compute.m` and
  `psi_compute.m`, run on them with FieldTrip's mtmconvol, return no delta-theta PSI at all and no
  power below 7 Hz in the baseline.
- `psianalysis.m` multiplies Fourier coefficients without the complex conjugate; its normalised
  cross-spectrum then depends on channel B's phase alone, up to sign, and is not the PSI of
  Nolte et al. This literal code reproduces the paper's alpha-beta PSI pattern.
- The channel order differs between subjects' files (13 subjects), with no note in the
  descriptor.
- Six subjects were recorded in two blocks with two trial tables whose trial numbers overlap;
  derived trials can be linked to them only by trial number with response time. For 4 of these
  patients the docs summary and Table S1 accuracies do not match the two tables.
- The pretrial segment's `fsample` field reads 255.64 Hz in 33 subjects though its time axis is
  256 Hz.


In [ ]:
print(f"notebook ran in {(time.time() - T0) / 60:.1f} min; peak memory of this process "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")